# 022 · Validation round on named cases

**What this notebook does.** The EDA (004) found what statistics can show: impossible values,
fixed in the pipeline (V2-22 to V2-24). A value can also look normal and still be wrong. This
notebook runs seven checks for such values over the whole build, sorts what each finds into
likely causes, and shows **named companies** next to their source (the SEC filing itself, or the
other price source), so that each doubt gets a verdict: no problem, a problem with a proposed
rule, or a decision for the researcher.

**Why now.** The models (005–007) come next. An error found after training means training again.

**How to read it.** Each section says what is checked and why, how big it is (in all rows and in
the rows the models learn from), the causes, named cases with evidence, and a verdict. The last
section is the verdict table and the findings.

**It changes nothing.** It reads `data/sentinel.duckdb` (read-only) and fetches SEC documents live
(with the contact from `.env`, under the SEC's rate limit); it writes nothing. The checks live in
`src/sentinel/validation.py` (tested). A check may look at later data to *find* a problem; any rule
proposed from it must use only what was known at the time (constitution I).

**Run it** (about 2 minutes, needs the internet for the SEC):
`systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/022-validation.ipynb`

In [ ]:
import time

import duckdb
import pandas as pd

from sentinel import validation as v

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.6g}".format)
con = duckdb.connect("../data/sentinel.duckdb", read_only=True, config={"memory_limit": "3GB"})
name = lambda cik: con.execute("select name from ref.company where cik = ?", [cik]).fetchone()[0]


def sec(cik, accession, words, width=250):
    time.sleep(0.2)  # well under the SEC's 10 requests a second
    url = v.filing_url(cik, accession)
    print(url)
    print(v.edgar_text(url, words, width)[:1500])

## 1. Yahoo and Tiingo disagree

**What.** For 6,841 companies we hold prices from both Yahoo and Tiingo. The pipeline's check
(`core.price_check`) calls a pair a *mismatch* when their daily returns differ by more than 2
points on 10% or more of the shared days. **Why it matters.** One series is chosen per company;
if the two disagree, the chosen one may be wrong. **How the causes are sorted:** by comparing the
two *closes* (prices as traded): closes agreeing on 90% of days means only the dividend or split
adjustment differs; a steady whole-number ratio (2, 3, 10, …) means one source states prices in
other units or after a split it applies differently (Yahoo's `close` is already split-adjusted,
Tiingo's is not); closes apart on most days may be a different stock under the same ticker.

In [ ]:
m = v.price_mismatch_causes(con)
print(len(m), "companies")
display(pd.crosstab(m["cause"], m["chosen"], margins=True))
cases = m[m["cause"].str.startswith("closes apart")].sort_values("differ_share", ascending=False).head(3)
cases = cases.assign(name=[name(c) for c in cases["cik"]])
display(cases[["cik", "name", "yahoo_ticker", "tiingo_ticker", "days", "differ_share", "close_ratio", "same_close", "chosen"]])

**Named cases:** the two sources' closes, a year at a time (median ratio of Yahoo to Tiingo).

In [ ]:
for cik, yt, tt in cases[["cik", "yahoo_ticker", "tiingo_ticker"]].itertuples(index=False):
    print(cik, name(cik), yt, tt)
    display(con.execute("""
        select year(y.session) as year, count(*) as days, median(y.close) as yahoo_close,
               median(t.close) as tiingo_close, median(y.close / t.close) as ratio
        from stg_yahoo.price y join stg_tiingo.price t on t.session = y.session
        where y.ticker = ? and t.ticker = ? and t.close > 0 group by 1 order by 1""", [yt, tt]).df())

## 2. One-month returns above +1,000%

**What.** Month-end rows (to 2022) whose one-month return is above +1,000%, with the biggest
one-day jump in that month and whether either source recorded a split within five days of it.
**Why.** A tenfold month can be real (a tiny company's drug approval), but a jump of thousands of
times in one day is not a price: it is a placeholder price before a listing, a unit error, or a
split the series misses. Such a day poisons every return-based feature and the volatility answer
for months.

In [ ]:
e = v.extreme_returns(con)
print(len(e), "rows;", int(e["eligible_volatility"].sum()), "eligible for volatility")
display(e[["yahoo_split", "tiingo_split"]].value_counts().rename("rows").to_frame())
print("one-day jump of 1,000x or more:", int((e["jump"] >= 1000).sum()), "rows")
d = con.execute("""
    with d as (select cik, session, close, adj_close / lag(adj_close) over w as r from core.price_daily
               window w as (partition by cik order by session))
    select count(*) as days, count(distinct cik) as companies,
           count(*) filter (where r >= 1000) as up_1000x, count(*) filter (where r <= 0.001) as down_1000x
    from d where r >= 1000 or r <= 0.001""").df()
display(d)
print("days with a close of $0.00001 or less:", con.execute(
    "select count(*), count(distinct cik) from core.price_daily where close <= 0.00001").fetchone())
print("volatility answers with a 1,000x day inside their window:", con.execute("""
    with d as (select cik, session, adj_close / lag(adj_close) over w as r from core.price_daily
               window w as (partition by cik order by session)),
    j as (select cik, session from d where r >= 1000 or r <= 0.001)
    select count(*) from label.volatility l where l.vol_63_future is not null and exists
      (select 1 from j where j.cik = l.cik and j.session > l.as_of and j.session <= l.window_end)""").fetchone()[0])
cases = e[e["eligible_volatility"]].head(3).assign(name=lambda x: [name(c) for c in x["cik"]])
display(cases[["cik", "name", "as_of", "ret_1m", "jump_day", "jump", "ticker", "yahoo_split", "tiingo_split"]])

**Named cases:** the chosen series around the jump, and the company's filings that month.

In [ ]:
for cik, day in cases[["cik", "jump_day"]].itertuples(index=False):
    print(cik, name(cik))
    display(con.execute("""select session, source, ticker, close, adj_close, split_ratio from core.price_daily
        where cik = ? and session between ?::DATE - 6 and ?::DATE + 4 order by session""", [cik, day, day]).df())
    display(con.execute("""select form, filing_date, items from core.filing where cik = ?
        and filing_date between ?::DATE - 60 and ?::DATE + 10 order by filing_date""", [cik, day, day]).df())

## 3. A split inside a dilution window

**What.** Dilution answers of 1 whose two share counts (the ones the label compares: the last
filed by the month-end, the last filed within 182 days) lie on both sides of a split in the price
series. The label adjusts the earlier count by the split. **Why.** If the counts and the price
series disagree about when the split happened, the adjustment itself creates a "dilution" of
exactly the split factor. The check flags answers whose increase equals the split factor, or whose
raw count did not change. **But:** a company can reverse-split and then sell many new shares (a
real dilution that leaves the count where it was), so every flagged case needs its filings read.

In [ ]:
s = v.split_dilution(con)
print(len(s), "dilution answers straddle a split;", int(s["split_read_as_dilution"].sum()),
      "flagged, at", s.loc[s["split_read_as_dilution"], "cik"].nunique(), "companies")
flag = s[s["split_read_as_dilution"]].drop_duplicates("cik").head(4)
display(flag.assign(name=[name(c) for c in flag["cik"]]))

**Named cases:** each company's accepted share counts around the window, with the filing each
came from, and the share count stated in that filing's text.

In [ ]:
for cik, as_of in flag[["cik", "as_of"]].itertuples(index=False):
    start = (pd.Timestamp(as_of) - pd.Timedelta(days=200)).date()
    end = (pd.Timestamp(as_of) + pd.Timedelta(days=200)).date()
    h = v.share_history(con, cik, str(start), str(end))
    print(cik, name(cik)); display(h)
    for acc in h["accession"].dropna().unique()[:3]:
        sec(cik, acc, r"(reverse (stock )?split|shares of .{0,60}common stock.{0,80}outstanding)", 200)

## 4. A company's first share count

**What.** Companies whose first accepted share count is 100 times or more away from the median of
their next three. **Why.** The typo rule (V2-23) compares a count with the company's last accepted
one, or with the same filing's average count; a company's very first count, without an average
count in its filing, has nothing to be compared with.

In [ ]:
f = v.first_count_outliers(con)
print(len(f), "companies; first count below 1,000 shares:", int((f["first_count"] < 1000).sum()),
      "| above the next three:", int((f["ratio"] > 1).sum()))
cases = f.head(3).assign(name=lambda x: [name(c) for c in x["cik"]])
display(cases)
for cik, as_of in cases[["cik", "as_of"]].itertuples(index=False):
    h = v.share_history(con, cik, str(as_of), str((pd.Timestamp(as_of) + pd.Timedelta(days=400)).date()))
    print(cik, name(cik)); display(h.head(4))
    acc = h["accession"].dropna()
    if len(acc):
        sec(cik, acc.iloc[0], r"shares of .{0,60}(common stock|ordinary shares).{0,80}outstanding", 200)

## 5. Five or more results releases in a year

**What.** Company-years with five or more results releases (four is normal), with the shortest
gap between two of them. **Why.** A release and its supplement counted twice would put two rows in
the results-move set for one report. The pipeline merges announcements within 5 days; the check
shows what lies beyond.

In [ ]:
r = v.crowded_releases(con)
print(len(r), "company-years; shortest gaps:", r["shortest_gap_days"].describe()[["min", "50%", "max"]].to_dict())
cases = r.head(3).assign(name=lambda x: [name(c) for c in x["cik"]])
display(cases[["cik", "name", "year", "releases", "shortest_gap_days"]])
for cik, year in cases[["cik", "year"]].itertuples(index=False):
    display(con.execute("""select form, filing_date, report_date, items from core.filing where cik = ?
        and year(filing_date) = ? and (form = '8-K' and items like '%2.02%' or form in ('10-Q', '10-K'))
        order by filing_date""", [cik, int(year)]).df())

## 6. Tiingo prices far from their own level

**What.** Companies priced from Tiingo (mostly ones that have disappeared) whose latest close is
10 times or more above or below the median close of their year before. **Why.** The app shows the
latest close; the volatility answer uses the last months of prices; a placeholder price at the
end of a series makes both absurd.

In [ ]:
t = v.tiingo_price_jumps(con)
print(len(t), "companies; last close of $0.00001 or less:", int((t["last_close"] <= 0.00001).sum()))
cases = t.head(4).assign(name=lambda x: [name(c) for c in x["cik"]])
display(cases)
for cik, day in cases[["cik", "last_day"]].itertuples(index=False):
    display(con.execute("""select session, close, volume from core.price_daily where cik = ?
        and session <= ? order by session desc limit 6""", [cik, day]).df().assign(cik=cik))

## 7. Insider lines no rule can judge

**What.** Insider trade lines worth more than \$1 billion that the V2-24 rule could not judge,
because no price of the stock was known by the filing date (before a listing, or with no price
series). **Why.** Open-market lines feed the insider features and the tracker; a typo there shows
as a billion-dollar trade.

In [ ]:
u = v.unjudged_insider_lines(con)
print(len(u), "lines,", u["cik"].nunique(), "companies; open-market:", int(u["open_market"].sum()),
      "at", u.loc[u["open_market"], "cik"].nunique(), "companies")
om = u[u["open_market"]].head(4)
om = om.assign(name=[name(c) for c in om["cik"]])
display(om)
for cik, filed in om[["cik", "filing_date"]].itertuples(index=False):
    print(cik, name(cik), "shares outstanding known by the filing date:", con.execute(
        "select shares, as_of from core.shares_outstanding where cik = ? and known_at <= ? order by known_at desc limit 1",
        [cik, filed]).fetchall())

## 8. Verdicts and findings

*Build of 30 October 2026. Each verdict rests on the named cases above, read against the SEC
filing or the other price source.*

| # | Check | Verdict | Size | Proposal |
|---|---|---|---|---|
| 1 | Yahoo and Tiingo disagree | **Problem**: for 17 companies the Tiingo series is another instrument with a similar ticker (warrants SQFTW, IHETW, SOUNW, HTZWW; preferred SQFTP, FGBIP; notes RWAYL, RWAYI; another class BELFB, BBXIA); for iHeartMedia and SoundHound that series is the chosen one | 17 companies (4 with Tiingo chosen); the other 177 mismatches are adjustment or period differences | V2-31 |
| 2 | One-month returns above +1,000% | **Problem**: placeholder prices, not splits. Tiingo shows \$0.0001 for days before a listing (Traws Pharma before its July 2013 IPO, Corporate Capital Trust before November 2017); a 198,800-fold "day" follows | 614 one-day moves of 1,000× or more at 213 companies; 333 volatility answers contain one | V2-28 |
| 3 | A split inside a dilution window | **Problem, partly**: the price series can date a split later than the filings (Medical Innovation Holdings' July 2016 10-Q already counts "post-reverse split 1-for-10" shares) or have a split the counts never show (LookSmart: 17,308,059 shares in three 2013 filings); the adjustment then makes a false 3× to 10× dilution. Blackstone Mortgage Trust's 2013 flag is a real dilution (a 1-for-10 reverse split, then a large share sale) | 241 flagged answers at 71 companies | V2-29 |
| 4 | A company's first share count | **Problem**: counts of 1 share: true for a holding company before its merger (TechnipFMC's 10-K: "outstanding as of January 13, 2017 was 1"), a tagging error elsewhere (Eco-Tek's 10-Q states 42,707,850 shares); unusable either way | 52 companies below 1,000 shares | V2-30 |
| 5 | Five or more results releases a year | **No problem**: real releases of companies that changed their fiscal year (CareFusion) or caught up on late reports (Faraday Future, Phoenix Companies) | 111 company-years | none |
| 6 | Tiingo prices far from their own level | **Problem**: bankrupt companies end on \$0.000001 with trading volume (EBIX, ViewRay, Accelerate Diagnostics); CLS Holdings sits at \$144,200 without trading | 399 companies, 48 ending at \$0.00001 or less; 20,796 such days at 236 companies | V2-28 |
| 7 | Insider lines no rule can judge | **Problem, small**: prices typed in millions for a stock that had no market price yet (Gobi Acquisition's \$200 million a share in 2021; Assured Guaranty 2008); 429 of the 796 open-market lines are from before 2010 | 143 open-market lines above \$1 million a share at 76 companies (51 already caught) | V2-32 |

### In plain words

The share counts and filings are mostly right; the weak points are **prices from Tiingo for
small, new or failed stocks**: placeholder prices before a listing and after a bankruptcy, and a
few series that belong to a warrant or a preferred share instead of the stock. They matter most
for the volatility answers (333 of them contain an impossible one-day move) and for the app's
latest prices. The dilution answer has one real weakness: the split adjustment trusts the price
series' date of a split over the filings.

### Proposed rules (each in `docs/decisions.md`, for the researcher)

- **V2-28 Placeholder prices**: a close of \$0.0001 or less is not a price; a one-day move of
  1,000× or more, up or down, that no split in either source explains is not a return. Such days
  are left out of every return-based feature and answer, as glitch days already are. Judged on the
  day itself: point-in-time.
- **V2-29 Splits in the dilution answer**: the split is applied at the step where the share counts
  themselves move by the split factor; if no step in the window does, the counts are taken as
  already comparable and no adjustment is made. Keeps Blackstone Mortgage Trust's real dilution,
  removes Medical Innovation's and LookSmart's false ones.
- **V2-30 Share counts below 1,000** are not used (no listed company has so few shares).
- **V2-31 Another instrument**: a Tiingo series whose closes differ from Yahoo's on most shared
  days, or whose ticker is the stock's ticker plus a warrant, preferred, note or class suffix, is
  not matched to the company.
- **V2-32 Insider prices above \$1 million a share** have no dollar value (no US stock has traded
  that high; Berkshire Hathaway's class A peaked near \$740,000).

After the researcher decides, the accepted rules go into the pipeline together, with tests, then
one rebuild, and the EDA and this notebook are re-run.